In [2]:
import pandas as pd

In [4]:
races = pd.read_csv('../data/raw/races.csv')
races.head()

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [6]:
results = pd.read_csv('../data/raw/results.csv')
drivers = pd.read_csv('../data/raw/drivers.csv')
constructors = pd.read_csv('../data/raw/constructors.csv')
driver_standings = pd.read_csv('../data/raw/driver_standings.csv')
constructor_standings = pd.read_csv('../data/raw/constructor_standings.csv')
status = pd.read_csv('../data/raw/status.csv')

In [8]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [10]:
results['position'].unique()

<StringArray>
[ '1',  '2',  '3',  '4',  '5',  '6',  '7',  '8', '\N',  '9', '10', '11', '12',
 '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
 '26', '27', '28', '29', '30', '31', '32', '33']
Length: 34, dtype: str

In [12]:
results['position_numeric'] = pd.to_numeric(results['position'], errors='coerce')
results['position_numeric'].unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8., nan,  9., 10., 11., 12.,
       13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
       26., 27., 28., 29., 30., 31., 32., 33.])

In [14]:
season_races = races[races['year'] == 2021]
season_races[['raceId', 'round', 'name', 'date']]

,raceId,round,name,date
1035,1053,2,Emilia Romagna Grand Prix,2021-04-18
1037,1052,1,Bahrain Grand Prix,2021-03-28
1038,1051,20,Qatar Grand Prix,2021-11-21
1039,1054,3,Portuguese Grand Prix,2021-05-02
1040,1055,4,Spanish Grand Prix,2021-05-09
1041,1056,5,Monaco Grand Prix,2021-05-23
1042,1057,6,Azerbaijan Grand Prix,2021-06-06
1043,1058,8,Styrian Grand Prix,2021-06-27
1044,1059,7,French Grand Prix,2021-06-20
1045,1060,9,Austrian Grand Prix,2021-07-04


In [18]:
season_results = results[results['raceId'].isin(season_races['raceId'])]
season_results.shape

(440, 19)

In [20]:
driver_points = (
    season_results
    .groupby('driverId')['points']
    .sum()
    .reset_index()
    .sort_values('points', ascending=False)
)
driver_points.head(10)

,driverId,points
8,830,388.5
0,1,385.5
7,822,219.0
5,815,190.0
9,832,163.5
15,846,160.0
14,844,159.0
6,817,114.0
13,842,110.0
1,4,81.0


In [22]:
driver_points_named = driver_points.merge(
    drivers[['driverId', 'forename', 'surname']],
    on='driverId'
)
driver_points_named.head(10)

,driverId,points,forename,surname
0,830,388.5,Max,Verstappen
1,1,385.5,Lewis,Hamilton
2,822,219.0,Valtteri,Bottas
3,815,190.0,Sergio,Pérez
4,832,163.5,Carlos,Sainz
5,846,160.0,Lando,Norris
6,844,159.0,Charles,Leclerc
7,817,114.0,Daniel,Ricciardo
8,842,110.0,Pierre,Gasly
9,4,81.0,Fernando,Alonso


In [24]:
last_race_id = season_races.sort_values('round')['raceId'].iloc[-1]

official = driver_standings[driver_standings['raceId'] == last_race_id]

check = driver_points.merge(
    official[['driverId', 'points']],
    on='driverId',
    suffixes=('_computed', '_official')
)
check['difference'] = check['points_computed'] - check['points_official']
check.sort_values('difference', ascending=False)

,driverId,points_computed,points_official,difference
3,815,190.0,190.0,0.0
9,4,81.0,81.0,0.0
8,842,110.0,110.0,0.0
6,844,159.0,159.0,0.0
5,846,160.0,160.0,0.0
17,841,3.0,3.0,0.0
14,847,16.0,16.0,0.0
15,8,10.0,10.0,0.0
16,849,7.0,7.0,0.0
11,20,43.0,43.0,0.0


In [26]:
sprint_results = pd.read_csv('../data/raw/sprint_results.csv')
sprint_results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1


In [28]:
season_sprint_results = sprint_results[sprint_results['raceId'].isin(season_races['raceId'])]

sprint_points = (
    season_sprint_results
    .groupby('driverId')['points']
    .sum()
    .reset_index()
    .rename(columns={'points': 'sprint_points'})
)

sprint_points.head()

,driverId,sprint_points
0,1,2
1,4,0
2,8,0
3,9,0
4,20,0


In [30]:
driver_points_full = driver_points.merge(
    sprint_points,
    on='driverId',
    how='left'
)

driver_points_full['sprint_points'] = driver_points_full['sprint_points'].fillna(0)

driver_points_full['total_points'] = (
    driver_points_full['points'] + driver_points_full['sprint_points']
)

driver_points_full.sort_values('total_points', ascending=False).head(10)

,driverId,points,sprint_points,total_points
0,830,388.5,7,395.5
1,1,385.5,2,387.5
2,822,219.0,7,226.0
3,815,190.0,0,190.0
4,832,163.5,1,164.5
5,846,160.0,0,160.0
6,844,159.0,0,159.0
7,817,114.0,1,115.0
8,842,110.0,0,110.0
9,4,81.0,0,81.0


In [32]:
check_full = driver_points_full.merge(
    official[['driverId', 'points']],
    on='driverId',
    suffixes=('_computed', '_official')
)
check_full['difference'] = check_full['total_points'] - check_full['points_official']
check_full.sort_values('difference', ascending=False)

,driverId,points_computed,sprint_points,total_points,points_official,difference
0,830,388.5,7,395.5,395.5,0.0
1,1,385.5,2,387.5,387.5,0.0
2,822,219.0,7,226.0,226.0,0.0
3,815,190.0,0,190.0,190.0,0.0
4,832,163.5,1,164.5,164.5,0.0
5,846,160.0,0,160.0,160.0,0.0
6,844,159.0,0,159.0,159.0,0.0
7,817,114.0,1,115.0,115.0,0.0
8,842,110.0,0,110.0,110.0,0.0
9,4,81.0,0,81.0,81.0,0.0


In [34]:
def get_driver_standings(year, races, results, sprint_results, drivers):
    season_races = races[races['year'] == year]
    season_results = results[results['raceId'].isin(season_races['raceId'])]

    points = (
        season_results
        .groupby('driverId')['points']
        .sum()
        .reset_index()
    )

    season_sprint = sprint_results[sprint_results['raceId'].isin(season_races['raceId'])]
    sprint_points = (
        season_sprint
        .groupby('driverId')['points']
        .sum()
        .reset_index()
        .rename(columns={'points': 'sprint_points'})
    )

    combined = points.merge(sprint_points, on='driverId', how='left')
    combined['sprint_points'] = combined['sprint_points'].fillna(0)
    combined['total_points'] = combined['points'] + combined['sprint_points']

    combined = combined.merge(
        drivers[['driverId', 'forename', 'surname']],
        on='driverId'
    )

    return combined.sort_values('total_points', ascending=False).reset_index(drop=True)

In [36]:
standings_2021 = get_driver_standings(2021, races, results, sprint_results, drivers)
standings_2021.head(10)

,driverId,points,sprint_points,total_points,forename,surname
0,830,388.5,7,395.5,Max,Verstappen
1,1,385.5,2,387.5,Lewis,Hamilton
2,822,219.0,7,226.0,Valtteri,Bottas
3,815,190.0,0,190.0,Sergio,Pérez
4,832,163.5,1,164.5,Carlos,Sainz
5,846,160.0,0,160.0,Lando,Norris
6,844,159.0,0,159.0,Charles,Leclerc
7,817,114.0,1,115.0,Daniel,Ricciardo
8,842,110.0,0,110.0,Pierre,Gasly
9,4,81.0,0,81.0,Fernando,Alonso


In [38]:
standings_2016 = get_driver_standings(2016, races, results, sprint_results, drivers)
standings_2016.head(10)

,driverId,points,sprint_points,total_points,forename,surname
0,3,385.0,0.0,385.0,Nico,Rosberg
1,1,380.0,0.0,380.0,Lewis,Hamilton
2,817,256.0,0.0,256.0,Daniel,Ricciardo
3,20,212.0,0.0,212.0,Sebastian,Vettel
4,830,204.0,0.0,204.0,Max,Verstappen
5,8,186.0,0.0,186.0,Kimi,Räikkönen
6,815,101.0,0.0,101.0,Sergio,Pérez
7,822,85.0,0.0,85.0,Valtteri,Bottas
8,807,72.0,0.0,72.0,Nico,Hülkenberg
9,4,54.0,0.0,54.0,Fernando,Alonso


In [42]:
def get_constructor_standings(year, races, results, sprint_results, constructors):
    season_races = races[races['year'] == year]
    season_results = results[results['raceId'].isin(season_races['raceId'])]

    points = (
        season_results
        .groupby('constructorId')['points']
        .sum()
        .reset_index()
    )

    season_sprint = sprint_results[sprint_results['raceId'].isin(season_races['raceId'])]
    sprint_points = (
        season_sprint
        .groupby('constructorId')['points']
        .sum()
        .reset_index()
        .rename(columns={'points': 'sprint_points'})
    )

    combined = points.merge(sprint_points, on='constructorId', how='left')
    combined['sprint_points'] = combined['sprint_points'].fillna(0)
    combined['total_points'] = combined['points'] + combined['sprint_points']

    combined = combined.merge(
        constructors[['constructorId', 'name']],
        on='constructorId'
    )

    return combined.sort_values('total_points', ascending=False).reset_index(drop=True)

In [43]:
constructor_standings_2021 = get_constructor_standings(2021, races, results, sprint_results, constructors)
constructor_standings_2021.head(10)

,constructorId,points,sprint_points,total_points,name
0,131,604.5,9,613.5,Mercedes
1,9,578.5,7,585.5,Red Bull
2,6,322.5,1,323.5,Ferrari
3,1,274.0,1,275.0,McLaren
4,214,155.0,0,155.0,Alpine F1 Team
5,213,142.0,0,142.0,AlphaTauri
6,117,77.0,0,77.0,Aston Martin
7,3,23.0,0,23.0,Williams
8,51,13.0,0,13.0,Alfa Romeo
9,210,0.0,0,0.0,Haas F1 Team
